# 05 - Vectorización con Apache Arrow y Pandas UDF

Las UDFs clásicas procesan fila por fila con alto consumo de CPU. Con **Apache Arrow**, los bloques de columnas se procesan vectorialmente en C++ mediante Pandas/NumPy.


In [ ]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

spark = get_spark_session("05_Arrow")
df_dep = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")

@pandas_udf(DoubleType())
def normalizar_altura(s: pd.Series) -> pd.Series:
    return s / 100.0

df_out = df_dep.withColumn("altura_m", normalizar_altura(F.col("altura")))
df_out.select("nombre", "altura", "altura_m").show(5)
